# 同一次计算，为什么能改按列来读？

按行计算能逐项得到输出，按列看却能追踪每个输入系数贡献了一整支怎样的向量。运行前按两种分组各算一次。拖动一个输入时，其输出变化应沿对应列，而不只是改变同编号的输出坐标。

从新内核按顺序运行；没有滑块时可直接修改参数。计算单元用于核对前面的推理，参考结果放在相应尝试之后。


In [ ]:
import numpy as np

A = np.array([[1, 2],
              [3, 1]])
x = np.array([4, -1])
b_out = A @ x
print('给定 A 和 x，算出的输出 b =', b_out)

## 把同一计算按输入分组

第一列是 `[1, 3]`，第二列是 `[2, 1]`。试着让输入中的 `4` 乘第一列、`−1` 乘第二列，再相加。会得到刚才的输出 `[2, 11]` 吗？如果题目改成“输出必须是 `[1, 7]`”，问题就变成找一个新的输入 `x`；那是解方程组，不是本格正在做的正向计算。

In [ ]:
a1 = A[:, 0]
a2 = A[:, 1]
from_columns = x[0] * a1 + x[1] * a2
print('第一列 =', a1)
print('第二列 =', a2)
print('按列组合 =', from_columns)
print('两种算法一致吗？', np.array_equal(b_out, from_columns))

## 拖动输入，观察输出如何移动

每个滑块控制一列取多少倍。先猜：只改变 `x1`，输出会沿哪个方向移动？拖动滑块验证。若当前环境没有 `ipywidgets`，本格会自动画出几组静态输入，数学结论仍然可以完整观察。

In [ ]:
import matplotlib.pyplot as plt

def show_combination(x1=1.0, x2=1.0):
    result = x1 * a1 + x2 * a2
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.quiver(0, 0, *a1, angles='xy', scale_units='xy', scale=1, color='tab:blue', label='a1')
    ax.quiver(0, 0, *a2, angles='xy', scale_units='xy', scale=1, color='tab:orange', label='a2')
    ax.quiver(0, 0, *result, angles='xy', scale_units='xy', scale=1, color='black', label='Ax')
    ax.set(xlim=(-5, 7), ylim=(-5, 7), xlabel='first output coordinate', ylabel='second output coordinate', aspect='equal')
    ax.axhline(0, color='0.75', linewidth=0.8)
    ax.axvline(0, color='0.75', linewidth=0.8)
    ax.grid(alpha=0.2)
    ax.legend()
    ax.set_title(f'Ax = {x1:.1f} a1 + {x2:.1f} a2 = {result}')
    plt.show()

try:
    from ipywidgets import interact, FloatSlider
except ImportError:
    print('Install ipywidgets for sliders; showing three parameter choices instead.')
    for values in [(1, 1), (2, 1), (1, -1)]:
        show_combination(*values)
else:
    interact(show_combination,
             x1=FloatSlider(value=1, min=-2, max=2, step=0.1),
             x2=FloatSlider(value=1, min=-2, max=2, step=0.1));

## 从例子抽出规则

`x1` 改变时，输出沿第一列方向变化；`x2` 改变时，输出沿第二列方向变化。对任意矩阵 `A = [a1 ... an]`，这就推广为 `Ax = x1 a1 + ... + xn an`。

下一问自然出现：这些列能组合出的所有输出，究竟构成什么集合？